In [1]:
import os
from openai import OpenAI
from IPython.display import display, Code, Markdown

# #硅基流动API
# ds_api_key = "sk-atisrejlfxlvnymvfxoesps"
# client = OpenAI(api_key=ds_api_key, 
#                 base_url="https://api.siliconflow.cn/v1")

ds_api_key = open('./ken_files/deepseekAPI-Key.md').read()
client = OpenAI(api_key=ds_api_key,
               base_url ='https://api.deepseek.com')

In [2]:
# 打开并读取Markdown文件
with open('./data/LC数据字典.md', 'r', encoding='utf-8') as f:
    md_content = f.read()
    
len(md_content)

1606

In [3]:
#基于md_content作为模型背景信息，向模型进行相关提问
response = client.chat.completions.create(
    # model="deepseek-ai/DeepSeek-V2.5", 
    model="deepseek-chat",  
    messages=[
        {"role": "system", "content": md_content}, 
        # "content": '请帮我统计下LC数据表一共有哪些字段？共计多少个？'
        {"role": "user", "content": '请帮我介绍下LC数据表'}
    ],
)
display(Markdown(response.choices[0].message.content))

# LC数据表介绍

## 基本概况

LC数据表是拍拍贷互联网金融公司在2015年1月1日至2017年1月30日期间记录的贷款用户信息表，包含30余万条贷款记录。该表全面记录了借款用户的多维度信息，包括：

- 用户基本信息（年龄、性别）
- 认证信息（户口认证、征信认证）
- 信用信息（初始评级、历史还款记录）
- 贷款信息（借款金额、借款类型、待还本金）

## 主要用途

该数据表主要用于：
1. 分析逾期用户的特征和行为模式
2. 识别高风险贷款用户群体
3. 优化信用评级模型
4. 制定风险控制策略
5. 提高公司业务收入并降低逾期风险

## 数据特点

- **高质量**：数据由拍拍贷平台直接采集，并通过回访确认，准确性和可信度高
- **时间跨度**：覆盖2年时间段的贷款数据
- **维度丰富**：包含用户身份、信用、行为等多方面信息
- **业务相关**：直接反映公司核心贷款业务情况

## 字段说明

表中包含11个字段，主要分为以下几类：

1. **标识信息**：序号（唯一标识）
2. **贷款信息**：借款金额、借款类型、初始评级
3. **用户信息**：年龄、性别
4. **认证信息**：户口认证、征信认证
5. **信用历史**：总待还本金、历史正常还款期数、历史逾期还款期数

该数据集非常适合用于信用风险评估、用户行为分析和金融风控建模等应用场景。

In [11]:
def get_sql_result(sql_query):
    """
    查询数据库相关数据的函数
    :param sql_query: 必要参数，字符串类型，用于表示查询数据的sql语句；
    :return：sql_query表示的sql语句查询到的结果;
    """
    connection = pymysql.connect(
            host='39.108.60.205',  # 数据库地址
            user='testaidb',  # 数据库用户名
            passwd='aa123456',  # 数据库密码
            db='testaidb',  # 数据库名
            charset='utf8'  # 字符集选择utf8
        )
    
    try:
        with connection.cursor() as cursor:
            # SQL查询语句
            sql = sql_query
            cursor.execute(sql)

            # 获取查询结果
            results = cursor.fetchall()

    finally:
        connection.close()
    
    
    return json.dumps(results)

In [56]:
import inspect
import json
import os
from openai import OpenAI
import pymysql
from IPython.display import display, Code, Markdown

#用于自动生成外部函数描述信息
def auto_function_desc(function): #参数为外部函数对象
    #定义一个内部函数用于生成外部函数的完整描述信息
    def inner(function):
        function_description = inspect.getdoc(function)#外部函数的函数说明
        function_name = function.__name__ #外部函数名
        
        system_prompt = '以下是某的函数说明：%s' % function_description
        
        user_prompt = '根据这个函数的函数说明，请帮我创建一个JSON格式的字典，这个字典有如下5点要求,请你仔细阅读，并且务必遵从所有要求：\
                       1.字典总共有三个键值对；\
                       2.第一个键值对的Key是字符串name，value是该函数的名字：%s，也是字符串；\
                       3.第二个键值对的Key是字符串description，value是该函数的函数的功能说明，也是字符串；\
                       4.第三个键值对的Key是字符串parameters，value是一个JSON Schema对象，用于说明该函数的参数输入规范。\
                       5.输出结果必须是一个JSON格式的字典，并且一定不要任何前后修饰语句,务必参按照如下格式进行输出:%s' % (function_name,'{key:value}')
        
        # api_key = "xxx"
        # client = OpenAI(api_key=ds_api_key, 
        #         base_url="https://api.siliconflow.cn/v1")

        ds_api_key = open('./ken_files/deepseekAPI-Key.md').read()
        client = OpenAI(api_key=ds_api_key,
               base_url ='https://api.deepseek.com')
        response = client.chat.completions.create(
        # model="deepseek-ai/DeepSeek-V2.5",  
        # model="deepseek-chat",  
        model="deepseek-reasoner",  
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}

             ]
        )

        return json.loads(response.choices[0].message.content)
    #由于模型根据提示信息生成的外部函数完整信息可能会有问题，因此，如果出现问题则loads环节会报错，则要求模型重新进行生成
    max_try_count = 5 #模型调用的最大次数
    count = 0 #当前调用模型的次数
    while count < max_try_count:
        try:
            function_desc = inner(function)
            break
        except Exception as e:
            count += 1
            print('something error:',e)
            if count == max_try_count:
                print('模型达到最大尝试次数，程序停止！')
                raise
            else:
                print('模型重新生成中......')
    tools = [
    {
        "type": "function", 
        "function":function_desc
    }
]
    return tools

In [30]:
desc_3 = auto_function_desc(get_sql_result)
desc_3

[{'type': 'function',
  'function': {'name': 'get_sql_result',
   'description': '查询数据库相关数据的函数',
   'parameters': {'type': 'object',
    'properties': {'sql_query': {'type': 'string',
      'description': '用于表示查询数据的sql语句'}},
    'required': ['sql_query']}}}]

In [49]:
# tools = [
#     {
#         "type":"fuction",
#         "fuction":desc_3
#     }
# ]

In [57]:
#available_functions表示外部函数库
def auto_run_conversation(messages,available_functions=None):
    # api_key = "sk-atisrrfnrxsnulmuvlzqnvuvcglkriejlfxlvnymvfxoesps"
    # client = OpenAI(api_key=ds_api_key, 
                # base_url="https://api.siliconflow.cn/v1")
    ds_api_key = open('./ken_files/deepseekAPI-Key.md').read()
    client = OpenAI(api_key=ds_api_key,
               base_url ='https://api.deepseek.com')
    
    # 如果没有外部函数库，则执行普通的对话任务
    if available_functions == None:
        print('模型原生能力解决该提问.........')
        response = client.chat.completions.create(
            # model="deepseek-ai/DeepSeek-V2.5",  
            model="deepseek-chat",  
            messages=messages
        )
        final_response = response.choices[0].message.content
    else:
        #外部函数库定义
        available_functions = available_functions
        
       # #step_3:外部函数完整描述定义 + #step_4:tools参数值定义
        tools = auto_function_desc(available_functions['function'])

        #step_5:第一次模型调用
        response = client.chat.completions.create(
            # model="deepseek-ai/DeepSeek-V2.5",  
            # model="deepseek-chat",  
            model="deepseek-chat",  
            messages=messages,
            tools=tools,
        )
        response_message = response.choices[0].message
        
        #判断返回结果是否存在tool_calls，即判断是否需要调用外部函数来回答问题
        if response_message.tool_calls:
            print('function_calling解决该提问.........')
            sql = response_message.tool_calls[0].function.arguments
            print('生成的sql为：:',sql)
            choose = input('是否执行上述sql? y/n')
            if choose == 'n':
                print("您选择不执行sql语句，再见！")
                return
            #step_6:外部函数手动调用且获取调用结果
            fuction_to_call = available_functions['function'] #函数对象
            function_args = json.loads(response_message.tool_calls[0].function.arguments)#函数参数

            function_response = fuction_to_call(**function_args)#函数手动调用

            #step_7:向messages进行两次消息追加
            messages.append(response_message.model_dump())  
            messages.append({
                        "role": "tool",
                        "content": function_response,
                        "tool_call_id":response_message.tool_calls[0].id
                    })

            #step_8: 再次调用大模型
            second_response = client.chat.completions.create(
                # model="deepseek-ai/DeepSeek-V2.5",
                            # model="deepseek-chat",  
                model="deepseek-chat",  
                messages=messages)
            final_response = second_response.choices[0].message.content
        else:
            final_response = response_message.content
    return Markdown(final_response)

In [53]:
tools

[{'type': 'fuction',
  'fuction': [{'type': 'function',
    'function': {'name': 'get_sql_result',
     'description': '查询数据库相关数据的函数',
     'parameters': {'type': 'object',
      'properties': {'sql_query': {'type': 'string',
        'description': '用于表示查询数据的sql语句'}},
      'required': ['sql_query']}}}]}]

In [ ]:
#测试

In [55]:
md_content

'# 数据库数据字典\n\n本数据字典记录了db001数据库中LC数据表的基本情况。\n\n## LC数据表\n\n- 基本解释\n\n   \tLC数据表记录了拍拍贷互联网金融公司在2015年1月1日到2017年1月30日期间共计30余万条的贷款用户的相关信息。用户的信息维度比较广泛，大致可分为用户的基本信息、认证信息、信用信息和贷款信息等。\n   \t\n   \t希望通过该数据表中的数据来分析得出逾期用户的特征，及公司核心业务标的，达到降低业务逾期风险，增加业务收入的目的。\n\n- 数据来源\n\n  \tLC数据集由拍拍贷平台进行的采集和记录，并且通过回访确认相关信息，数据集的准确性和可信度都非常高。\n\n- 各字段说明\n\n| Column Name      | Description                                        | Value Range               | Type         |\n| ---------------- | -------------------------------------------------- | ------------------------- | ------------ |\n| 序号             | 贷款用户的唯一标识                                 |                           | INT          |\n| 借款金额         | 借款成交总金额                                     |                           | FLOAT        |\n| 初始评级         | 借款成交时的信用评级                               | A,B,C,D,E,F               | VARCHAR(255) |\n| 借款类型         | 借款的具体类型                                     | 电商，APP闪电，普通和其他 | VARCHAR(255) |\n| 年龄             | 借款人在借款

In [58]:
messages = [
    {"role": "system", "content": md_content},
    {"role": "user", "content": "请问LC数据表有多少男性用户？"}
]
#定义外部函数库
available_functions = {
            "function": get_sql_result,
        }
auto_run_conversation(messages,available_functions)

function_calling解决该提问.........
生成的sql为：: {"sql_query":"SELECT COUNT(*) AS male_count FROM LC WHERE 性别 = '男';"}


是否执行上述sql? y/n y


LC数据表中共有507名男性用户。

In [ ]:
#調用二

In [59]:
messages = [
    {"role": "system", "content": md_content},
    {"role": "user", "content": "请问LC数据表有多少女性用户？"}
]
#定义外部函数库
available_functions = {
            "function": get_sql_result,
        }
auto_run_conversation(messages,available_functions)

something error: Expecting value: line 1 column 1 (char 0)
模型重新生成中......
something error: Expecting value: line 1 column 1 (char 0)
模型重新生成中......
something error: Expecting value: line 1 column 1 (char 0)
模型重新生成中......
function_calling解决该提问.........
生成的sql为：: {"sql_query":"SELECT COUNT(*) FROM LC WHERE 性别 = '女'"}


是否执行上述sql? y/n y


LC数据表中有 **200** 位女性用户。

In [ ]:
#调用三

In [60]:
messages = [
    {"role": "system", "content": md_content},
    {"role": "user", "content": "请问LC数据表不同年龄的用户数量是多少？"}
]
#定义外部函数库
available_functions = {
            "function": get_sql_result,
        }
auto_run_conversation(messages,available_functions)

function_calling解决该提问.........
生成的sql为：: {"sql_query":"SELECT 年龄, COUNT(*) AS 用户数量 FROM LC GROUP BY 年龄 ORDER BY 年龄;"}


是否执行上述sql? y/n y


以下是LC数据表中不同年龄段的用户数量分布：

- 19岁: 2人
- 20岁: 6人
- 21岁: 15人
- 22岁: 27人
- 23岁: 49人
- 24岁: 53人
- 25岁: 68人
- 26岁: 57人
- 27岁: 59人
- 28岁: 59人
- 29岁: 37人
- 30岁: 35人
- 31岁: 34人
- 32岁: 24人
- 33岁: 26人
- 34岁: 19人
- 35岁: 28人
- 36岁: 21人
- 37岁: 12人
- 38岁: 16人
- 39岁: 8人
- 40岁: 9人
- 41岁: 7人
- 42岁: 10人
- 43岁: 4人
- 44岁: 6人
- 45岁: 3人
- 46岁: 2人
- 47岁: 3人
- 49岁: 2人
- 50岁: 1人
- 51岁: 1人
- 52岁: 1人
- 53岁: 2人
- 54岁: 1人

从数据可以看出，25-28岁年龄段的用户数量最多，其中25岁用户最多(68人)，27岁和28岁用户数量相同(59人)。随着年龄增大或减小，用户数量都呈现递减趋势。

In [ ]:
# 调用四

In [61]:
messages = [
    {"role": "system", "content": md_content},
    {"role": "user", "content": "请问LC数据表将年龄进行分段处理，20-30,30-40,40-50，计算不同年龄段的用户数量？？"}
]
#定义外部函数库
available_functions = {
            "function": get_sql_result,
        }
auto_run_conversation(messages,available_functions)

something error: Expecting value: line 1 column 1 (char 0)
模型重新生成中......
function_calling解决该提问.........
生成的sql为：: {"sql_query":"SELECT CASE WHEN 年龄 BETWEEN 20 AND 30 THEN '20-30' WHEN 年龄 BETWEEN 30 AND 40 THEN '30-40' WHEN 年龄 BETWEEN 40 AND 50 THEN '40-50' ELSE '其他' END AS 年龄段, COUNT(*) AS 用户数量 FROM LC GROUP BY 年龄段;"}


是否执行上述sql? y/n y


根据LC数据表中年龄分段统计结果如下：

各年龄段用户数量分布：
- 20-30岁：465人
- 30-40岁：197人
- 40-50岁：38人
- 其他年龄段：7人

从数据可以看出，20-30岁的年轻用户群体是主要的贷款人群，占比最大；随着年龄增长，用户数量逐渐减少。